#### Group Fairness 모델
- FairGNN (Adversarial Learning): 대표
    - FairVGNN (Adversarial Learning)

- EDITS (Edge Rewiring): 대표
    - UGE (Edge Rewiring) : 제외
    - FairEdit (Edge Rewiring)
    - GEAR (Edge Rewiring)

- FairWalk (Rebalancing): 대표
    - CrossWalk (Rebalancing)

- NIFTY (Optimization with Regularization)

- FairGB

In [1]:
import pandas as pd

### Avg.

In [ ]:
df1 = pd.read_csv('./outputs/exp_fairgate_gcn.csv')
df1 = df1[['model', 'dataset', 'acc_mean', 'roc_auc_mean', 'dp_mean', 'eo_mean']]

df2 = pd.read_csv('./outputs/compare/exp_baselines.csv')
df2 = df2[['model', 'dataset', 'acc_mean', 'roc_auc_mean', 'dp_mean', 'eo_mean']]

df = pd.concat([df1, df2])
df_rank_v1 = df[df["model"] != "GCN"].copy()  # baseline 제외

df_rank = df_rank_v1[df_rank_v1['dataset'] != 'recidivism'].copy() # recidivism 제외시
# df_rank = df_rank_v1.copy() # 제외 아닐 시

# dataset별 rank 계산
df_rank["rank_Acc"] = df_rank.groupby("dataset")["acc_mean"].rank(ascending=False, method="average")
df_rank["rank_AUC"] = df_rank.groupby("dataset")["roc_auc_mean"].rank(ascending=False, method="average")
df_rank["rank_DP"]  = df_rank.groupby("dataset")["dp_mean"].rank(ascending=True,  method="average")
df_rank["rank_EO"]  = df_rank.groupby("dataset")["eo_mean"].rank(ascending=True,  method="average")

# accuracy / fairness 평균 rank
df_rank["rank_acc_avg"] = df_rank[["rank_Acc", "rank_AUC"]].mean(axis=1)
df_rank["rank_fair_avg"] = df_rank[["rank_DP", "rank_EO"]].mean(axis=1)

# 전체 dataset에 대한 모델별 평균 rank
model_order = ['FairGNN', 'FairVGNN', 
               'FairEdit', 'EDITS', 
               'FairWalk', 'CrossWalk', 
               'FairGB', 'NIFTY', 
               'FairGT', 'FairGate']

summary = (
    df_rank.groupby("model")[["rank_Acc", "rank_AUC", "rank_DP", "rank_EO",
                              "rank_acc_avg", "rank_fair_avg"]]
    .mean()
)

summary = summary.reindex(model_order)

print(summary.round(4))

In [4]:
df1 = pd.read_csv('./outputs/ours/exp_fairgate_fiw_v1.csv')
# df1 = pd.read_csv('./outputs/exp_main.csv')
df1 = df1[['model', 'dataset', 'acc_mean', 'roc_auc_mean', 'dp_mean', 'eo_mean']]

df2 = pd.read_csv('./outputs/compare/exp_baselines.csv')
df2 = df2[['model', 'dataset', 'acc_mean', 'roc_auc_mean', 'dp_mean', 'eo_mean']]

df = pd.concat([df1, df2], ignore_index=True)


df_rank_v1 = df[df["model"] != "GCN"].copy()   # baseline 제외
# df_rank = df_rank_v1[df_rank_v1['dataset'] != 'recidivism'].copy() # recidivism 제외
df_rank = df_rank_v1.copy()


df_rank["rank_Acc"] = df_rank.groupby("dataset")["acc_mean"].rank(
    ascending=False, method="average"
)
df_rank["rank_AUC"] = df_rank.groupby("dataset")["roc_auc_mean"].rank(
    ascending=False, method="average"
)
df_rank["rank_DP"] = df_rank.groupby("dataset")["dp_mean"].rank(
    ascending=True, method="average"
)
df_rank["rank_EO"] = df_rank.groupby("dataset")["eo_mean"].rank(
    ascending=True, method="average"
)

# metric-group average rank
df_rank["rank_acc_avg"] = df_rank[["rank_Acc", "rank_AUC"]].mean(axis=1)
df_rank["rank_fair_avg"] = df_rank[["rank_DP", "rank_EO"]].mean(axis=1)


df_tmp = (
    df_rank.groupby(["model", "dataset"], as_index=False)[
        ["rank_Acc", "rank_AUC", "rank_DP", "rank_EO",
         "rank_acc_avg", "rank_fair_avg"]
    ]
    .mean()
)

summary = (
    df_tmp.groupby("model")[
        ["rank_Acc", "rank_AUC", "rank_DP", "rank_EO",
         "rank_acc_avg", "rank_fair_avg"]
    ]
    .mean()
)

# dataset coverage count
summary["num_datasets"] = df_tmp.groupby("model")["dataset"].nunique()

model_order = [
    'FairGNN', 'FairVGNN',
    'FairEdit', 'EDITS',
    'FairWalk', 'CrossWalk',
    'FairGB', 'NIFTY',
    'FairGT', 'FairGate'
]

summary = summary.reindex(model_order)

# =========================
# 6. Print
# =========================
print(summary.round(2))

           rank_Acc  rank_AUC  rank_DP  rank_EO  rank_acc_avg  rank_fair_avg  \
model                                                                          
FairGNN        2.11      5.44     7.56     7.22          3.78           7.39   
FairVGNN       5.56      3.67     5.22     5.67          4.61           5.44   
FairEdit       5.40      6.60     5.80     6.80          6.00           6.30   
EDITS          8.00      5.33     9.00     6.00          6.67           7.50   
FairWalk       6.78      8.00     3.44     4.67          7.39           4.06   
CrossWalk      6.44      7.78     4.33     5.17          7.11           4.75   
FairGB         3.56      2.00     5.11     3.94          2.78           4.53   
NIFTY          5.89      6.56     6.56     6.22          6.22           6.39   
FairGT         5.89      3.67     3.56     4.11          4.78           3.83   
FairGate       2.44      1.78     2.33     1.56          2.11           1.94   

           num_datasets  
model        

### ours

In [ ]:
file_name ='exp_fairgate_gcn'

datasets = [
    'pokec_z', 'pokec_n', 
    'pokec_z_g', 'pokec_n_g',
    'credit', 'recidivism', 'income', 
    'german', 
    'nba'
             ]

df = pd.read_csv(f'./outputs/{file_name}.csv')
df = df[[
    'model', 'dataset',
    'acc_mean', 
    'acc_std', 
    'roc_auc_mean', 
    'roc_auc_std',
    'dp_mean', 
    'dp_std', 
    'eo_mean', 
    'eo_std', 
    # 'time_sec_mean', 'time_sec_std'
]]

df = df.copy()
df['dataset'] = pd.Categorical(
    df['dataset'],
    categories=datasets,
    ordered=True
)

df.sort_values('dataset')
# [df.dataset == 'german']

In [ ]:
file_name ='exp_fairgate_fiw_v1'

datasets = ['pokec_z', 'pokec_n', 'pokec_z_g', 'pokec_n_g',
             'credit', 'recidivism', 'income', 'german', 'nba']

df = pd.read_csv(f'./outputs/ours/{file_name}.csv')
df = df[[
    'model', 'dataset',
    'acc_mean', 
    # 'acc_std', 
    'roc_auc_mean', 
    # 'roc_auc_std',
    'dp_mean', 
    # 'dp_std', 
    'eo_mean', 
    # 'eo_std', 
    # 'time_sec_mean', 'time_sec_std'
]]

df = df.copy()
df['dataset'] = pd.Categorical(
    df['dataset'],
    categories=datasets,
    ordered=True
)

df.sort_values('dataset')
# [df.dataset == 'nba']

In [2]:
df = pd.read_csv(f'./outputs/exp_main.csv')
df = df[[
    'model', 'dataset',
    'acc_mean', 
    # 'acc_std', 
    'roc_auc_mean', 
    # 'roc_auc_std',
    'dp_mean', 
    # 'dp_std', 
    'eo_mean', 
    # 'eo_std', 
    # 'time_sec_mean', 'time_sec_std'
]]
df


,model,dataset,acc_mean,roc_auc_mean,dp_mean,eo_mean
0,FairGate,pokec_z,0.6908,0.7563,0.0130,0.0085
1,FairGate,pokec_n,0.6982,0.7413,0.0394,0.0587
2,FairGate,pokec_z_g,0.6886,0.7518,0.0745,0.0192
3,FairGate,pokec_n_g,0.6689,0.7318,0.0270,0.0639
4,FairGate,credit,0.7941,0.7237,0.0216,0.0071
5,FairGate,recidivism,0.8229,0.8465,0.0609,0.0478
6,FairGate,income,0.8087,0.7672,0.0316,0.0102
7,FairGate,german,0.6808,0.6320,0.0155,0.0154
8,FairGate,nba,0.7296,0.7812,0.0293,0.0672


### baselines

In [ ]:
file_name ='exp_baselines'
df_v2 = pd.read_csv(f'./outputs/compare/{file_name}.csv')
df_v2 = df_v2[[
    'model', 'dataset',
    'acc_mean', 
    'acc_std', 
    'roc_auc_mean',
    'roc_auc_std',
    'dp_mean',
    'dp_std',
    'eo_mean', 
    'eo_std', 
    # 'time_sec_mean', 
    # 'time_sec_std'
]]

datasets = ['pokec_z', 'pokec_n', 'pokec_z_g', 'pokec_n_g',
             'credit', 'recidivism', 'income', 'german', 'nba']
model_order = ['FairGNN', 'FairVGNN', 'FairEdit', 'EDITS',
               'FairWalk', 'CrossWalk', 'FairGB', 'NIFTY', 'FairGT']

filtered_df = df_v2.copy()
filtered_df['model'] = pd.Categorical(
    filtered_df['model'],
    categories=model_order,
    ordered=True
)
filtered_df['dataset'] = pd.Categorical(
    filtered_df['dataset'],
    categories=datasets,
    ordered=True
)

filtered_df = filtered_df.sort_values(['model', 'dataset'])
# filtered_df[filtered_df.dataset == 'recidivism']
filtered_df

In [ ]:
file_name ='exp_baselines_sc'
df_v2 = pd.read_csv(f'./outputs/{file_name}.csv')
df_v2 = df_v2[[
    'model', 'dataset',
    'acc_mean', 
    # 'acc_std', 
    'roc_auc_mean',
    # 'roc_auc_std',
    'dp_mean',
    # 'dp_std',
    'eo_mean', 
    # 'eo_std', 
    # 'time_sec_mean', 
    # 'time_sec_std'
]]

datasets = ['pokec_z', 'pokec_n', 'pokec_z_g', 'pokec_n_g',
             'credit', 'recidivism', 'income', 'german', 'nba']
model_order = ['FairGNN', 'FairVGNN', 'FairEdit', 'EDITS',
               'FairWalk', 'CrossWalk', 'FairGB', 'NIFTY', 'FairGT']

filtered_df = df_v2.copy()
filtered_df['model'] = pd.Categorical(
    filtered_df['model'],
    categories=model_order,
    ordered=True
)
filtered_df['dataset'] = pd.Categorical(
    filtered_df['dataset'],
    categories=datasets,
    ordered=True
)

filtered_df = filtered_df.sort_values(['model', 'dataset'])
filtered_df[filtered_df.model == 'CrossWalk']
# filtered_df

### gcn

In [ ]:
file_name ='exp_gcn'
df_ab = pd.read_csv(f'./outputs/compare/{file_name}.csv')
# df_ab = df_ab[df_ab.dataset == 'recidivism']
df_ab = df_ab.drop(['stage_label', 'run', 'seed'], axis=1)

df_ab.groupby(['dataset', 'stage']).agg(['mean', 'std']).round(4)